# EzyZip: Owner-aware temporal monitoring of personal food/drink containers using edge AI to identify suspicious interactions and potential unauthorized access.

## Project Overview

An intelligent surveillance and security system designed to detect unauthorized tampering with food items (cups, bottles, lunch boxes). The system uses multiple AI models to:

1. **Identify the authenticated owner** via face recognition
2. **Detect hand gestures** and track hand movements
3. **Identify food containers** using object detection
4. **Alert on tampering** when unauthenticated individuals touch monitored items
5. **Log events and performance metrics** for security audit trails
6. **Send real-time alerts** via Telegram

### Key Features:
- Real-time multi-modal AI processing
- Owner authentication via face recognition (InsightFace)
- Hand pose tracking (MediaPipe)
- Food item detection (YOLOv8 Custom ONNX model)
- Tampering event logging and Telegram notifications
- Hardware performance monitoring (CPU, GPU, RAM)
- Event buffering and evidence capture

---

## Installation & Setup

### Install Dependencies

First, install all required packages from requirements.txt:

In [ ]:
# Run this cell to install all dependencies
# !pip install -r requirements.txt

# Core packages needed:
# - opencv-python>=4.8.0
# - mediapipe>=0.10.0
# - insightface>=0.7.3
# - onnxruntime>=1.15.0
# - psutil>=5.9.0
# - gputil (optional, for GPU monitoring)
# - python-telegram-bot (for alerts)
# - matplotlib & seaborn (for visualization)

print("Dependencies installation guide:")
print("Run: pip install -r requirements.txt")

### Download Pre-trained Models

The system uses pre-trained models from MediaPipe and a custom YOLOv8 model:

In [ ]:
import os
import urllib.request

def download_models():
    """Downloads AI models directly to the root folder."""
    models = {
        "hand_landmarker.task": "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task",
        "face_landmarker.task": "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task",
        "efficientdet.tflite": "https://storage.googleapis.com/mediapipe-tasks/object_detector/efficientdet_lite0_uint8.tflite"
    }
    for filename, url in models.items():
        if not os.path.exists(filename):
            print(f"Downloading {filename}... Please wait.")
            urllib.request.urlretrieve(url, filename)
        else:
            print(f"[OK] {filename} already exists.")

# Uncomment to download models
# download_models()
print("Models are pre-trained and already available in the project directory.")

---

## Module 1: Hand Tracking (MediaPipe)

### Purpose:
Detects and tracks hand landmarks in real-time using MediaPipe's HandLandmarker model. Supports up to 6 hands simultaneously.

### Key Components:
- **21 hand landmarks** per hand (finger joints, palm, etc.)
- **Hand connections** to draw skeletal structure
- **Real-time video processing** with timestamp synchronization

In [ ]:
import cv2
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

class HandTracker:
    """Detects and tracks hand landmarks using MediaPipe."""
    
    def __init__(self):
        options = vision.HandLandmarkerOptions(
            base_options=python.BaseOptions(model_asset_path='hand_landmarker.task'),
            running_mode=vision.RunningMode.VIDEO,
            num_hands=6  # Support up to 6 hands
        )
        self.detector = vision.HandLandmarker.create_from_options(options)
        
        # Hand skeleton connections (21 landmarks)
        self.connections = [
            (0, 1), (1, 2), (2, 3), (3, 4),       # Thumb
            (0, 5), (5, 6), (6, 7), (7, 8),       # Index finger
            (5, 9), (9, 10), (10, 11), (11, 12),  # Middle finger
            (9, 13), (13, 14), (14, 15), (15, 16),# Ring finger
            (13, 17), (0, 17), (17, 18), (18, 19), (19, 20)  # Pinky + palm
        ]

    def process_and_return(self, mp_image, timestamp_ms):
        """Process image and return hand landmarks in pixel coordinates."""
        h, w = mp_image.height, mp_image.width
        results = self.detector.detect_for_video(mp_image, timestamp_ms)
        
        hands_data = []
        if results.hand_landmarks:
            for hand_lms in results.hand_landmarks:
                # Convert normalized coordinates to pixel coordinates
                pixel_lms = [(int(lm.x * w), int(lm.y * h)) for lm in hand_lms]
                hands_data.append(pixel_lms)
        return hands_data

    def close(self):
        """Clean up resources."""
        self.detector.close()

print("HandTracker class loaded successfully!")
print("\nKey Methods:")
print("  - __init__(): Initialize with MediaPipe hand landmarker model")
print("  - process_and_return(mp_image, timestamp_ms): Detect hands and return landmarks")
print("  - close(): Clean up detector resources")

---

## Module 2: Face Recognition (InsightFace + MediaPipe)

### Purpose:
Authenticates the owner and detects intruders using dual-model approach:
- **MediaPipe**: Ultra-fast face detection and landmark localization
- **InsightFace**: Deep face embedding and similarity comparison

### Security Features:
- **Threading-based processing** to avoid camera lag
- **3-second cooldown** to prevent spam detection
- **Similarity threshold (0.40)** for authentic owner verification

In [ ]:
import os
import cv2
import numpy as np
import threading
import time
from insightface.app import FaceAnalysis
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

class FaceTracker:
    """Dual-model face authentication system."""
    
    def __init__(self, user_image_path="user.jpg"):
        # 1. Initialize MediaPipe (Ultra-fast foreground tracking)
        options = vision.FaceLandmarkerOptions(
            base_options=python.BaseOptions(model_asset_path='face_landmarker.task'),
            running_mode=vision.RunningMode.VIDEO,
            num_faces=1
        )
        self.detector = vision.FaceLandmarker.create_from_options(options)
        
        # 2. Initialize InsightFace
        print("\nInitializing InsightFace (buffalo_l model)...")
        self.app = FaceAnalysis(name='buffalo_l', providers=['CPUExecutionProvider'])
        # Optimization: Reduced internal detection resolution for faster CPU execution
        self.app.prepare(ctx_id=0, det_size=(320, 320))
        
        # 3. State Management Variables
        self.user_embedding = None
        self.current_label = "Scanning..."
        self.current_color = (0, 255, 255)  # Yellow
        self.is_authenticated = False
        
        # Threading controls
        self.recognition_thread = None
        self.is_processing_identity = False
        
        # Security Cooldown controls
        self.last_auth_time = 0
        self.auth_cooldown = 3.0  # Seconds to wait before re-verifying

        # Load Master Identity
        if os.path.exists(user_image_path):
            print(f"Loading identity from {user_image_path}...")
            user_img = cv2.imread(user_image_path)
            faces = self.app.get(user_img)
            if faces:
                self.user_embedding = faces[0].embedding
                print(f"[SUCCESS] Identity securely loaded into memory!")
            else:
                self.current_label = "BAD user.jpg"
                self.current_color = (150, 150, 150)
        else:
            print(f"[CRITICAL WARNING] '{user_image_path}' not found!")
            self.current_label = "NO user.jpg FOUND"
            self.current_color = (150, 150, 150)

    def _recognize_face(self, face_crop):
        """BACKGROUND TASK: Runs heavy math without freezing the webcam."""
        self.is_processing_identity = True
        try:
            detected_faces = self.app.get(face_crop)
                
            if detected_faces:
                current_embedding = detected_faces[0].embedding
                # Cosine similarity
                sim = np.dot(self.user_embedding, current_embedding) / (
                    np.linalg.norm(self.user_embedding) * np.linalg.norm(current_embedding)
                )
                
                if sim > 0.40:  # Threshold for InsightFace
                    self.current_label = f"Owner ({sim:.2f})"
                    self.current_color = (0, 255, 0)  # Green
                    self.is_authenticated = True
                else:
                    self.current_label = f"INTRUDER! ({sim:.2f})"
                    self.current_color = (0, 0, 255)  # Red
                    self.is_authenticated = False
            else:
                self.current_label = "Scan Failed. Retrying..."
                self.current_color = (0, 165, 255)  # Orange
                self.is_authenticated = False
        except Exception as e:
            print(f"Error in face recognition thread: {e}")
            self.current_label = "Scan Error"
            self.current_color = (0, 165, 255)
        finally:
            self.is_processing_identity = False

    def process_and_draw(self, img, clean_img, mp_image, timestamp_ms, frame_count):
        """Process frame and update authentication status."""
        h, w, _ = img.shape
        results = self.detector.detect_for_video(mp_image, timestamp_ms)
        face_detected_this_frame = False
        current_time = time.time()

        if results.face_landmarks:
            for face_lms in results.face_landmarks:
                face_detected_this_frame = True
                
                # Draw Eye points
                left_eye = [362, 382, 381, 380, 374, 373, 390, 249, 263, 466, 388, 387, 386, 385, 384, 398]
                right_eye = [33, 7, 163, 144, 145, 153, 154, 155, 133, 246, 161, 160, 159, 158, 157, 173]
                for idx in left_eye + right_eye:
                    pt = face_lms[idx]
                    cv2.circle(img, (int(pt.x * w), int(pt.y * h)), 2, (255, 255, 255), -1)

                # Calculate Bounding Box
                x_coords = [lm.x for lm in face_lms]
                y_coords = [lm.y for lm in face_lms]
                left = int((min(x_coords) - 0.1) * w)
                right = int((max(x_coords) + 0.1) * w)
                top = int((min(y_coords) - 0.25) * h)
                bottom = int((max(y_coords) + 0.1) * h)
                left, right = max(0, left), min(w - 1, right)
                top, bottom = max(0, top), min(h - 1, bottom)

                # --- EVENT-DRIVEN AUTHENTICATION ---
                if self.user_embedding is not None and not self.is_authenticated and not self.is_processing_identity:
                    if current_time - self.last_auth_time > self.auth_cooldown:
                        face_crop = clean_img[top:bottom, left:right]
                        
                        if face_crop.size != 0:
                            self.current_label = "Authenticating..."
                            self.current_color = (0, 255, 255)
                            self.last_auth_time = current_time
                            
                            # Dispatch to background thread
                            self.recognition_thread = threading.Thread(target=self._recognize_face, args=(face_crop,))
                            self.recognition_thread.start()
                
                # Draw the box and label
                cv2.rectangle(img, (left, top), (right, bottom), self.current_color, 2)
                cv2.putText(img, self.current_label, (left, top - 15), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.8, self.current_color, 2, cv2.LINE_AA)

        # Reset security state if person leaves camera view
        if not face_detected_this_frame:
            self.is_authenticated = False
            self.last_auth_time = 0
            if not self.is_processing_identity:
                self.current_label = "Scanning..."
                self.current_color = (0, 255, 255)

    def close(self):
        """Clean up resources."""
        self.detector.close()
        if self.recognition_thread is not None and self.recognition_thread.is_alive():
            self.recognition_thread.join(timeout=1.0)

print("FaceTracker class loaded successfully!")
print("\nKey Features:")
print("  - Dual-model authentication (MediaPipe + InsightFace)")
print("  - Background processing thread to avoid lag")
print("  - 3-second cooldown to prevent spam")
print("  - Color-coded labels (Green=Owner, Red=Intruder, Yellow=Scanning)")

---

## Module 3: Food Object Detection (YOLOv8 ONNX)

### Purpose:
Detects food containers (bottles, cups, lunch boxes) using a custom-trained YOLOv8 model in ONNX format for fast CPU inference.

### Key Features:
- **ONNX Runtime** for cross-platform inference
- **NMS (Non-Maximum Suppression)** to filter overlapping detections
- **Face intersection checking** to avoid false positives on faces
- **Confidence threshold (0.60)** for high-quality predictions

In [ ]:
import cv2
import numpy as np
import onnxruntime as ort

class FoodDetector:
    """YOLOv8 ONNX model for food container detection."""
    
    def __init__(self, model_path="best.onnx"):
        print("Initializing YOLOv8 Custom (ONNX)...")
        self.session = ort.InferenceSession(model_path, providers=['CPUExecutionProvider'])
        self.input_name = self.session.get_inputs()[0].name
        
        # Class mapping
        self.target_classes = {
            0: "0",
            1: "Bottle", 
            2: "Lunch Box",
            3: "cup"
        }
        self.valid_classes = ["Bottle", "Lunch Box", "cup"]

    def _calculate_intersection_area(self, box1, box2):
        """Calculate intersection area of two boxes."""
        x1 = max(box1[0], box2[0])
        y1 = max(box1[1], box2[1])
        x2 = min(box1[0] + box1[2], box2[0] + box2[2])
        y2 = min(box1[1] + box1[3], box2[1] + box2[3])
        
        if x2 < x1 or y2 < y1:
            return 0.0
        return (x2 - x1) * (y2 - y1)

    def process_and_draw(self, img, clean_img, face_boxes=None):
        """Detect food containers in image."""
        if face_boxes is None:
            face_boxes = []

        img_h, img_w = clean_img.shape[:2]
        
        # Prepare input
        rgb_img = cv2.cvtColor(clean_img, cv2.COLOR_BGR2RGB)
        input_img = cv2.resize(rgb_img, (640, 640))
        input_img = input_img.astype(np.float32) / 255.0
        input_img = input_img.transpose(2, 0, 1)  # HWC -> CHW
        input_tensor = np.expand_dims(input_img, axis=0)

        # Run inference
        outputs = self.session.run(None, {self.input_name: input_tensor})[0]
        predictions = np.squeeze(outputs).T
        
        # Parse output
        boxes = predictions[:, :4]  # cx, cy, w, h
        scores = predictions[:, 4:]
        class_ids = np.argmax(scores, axis=1)
        confidences = np.max(scores, axis=1)
        
        # Filter by confidence and class
        mask = (confidences > 0.60) & np.isin(class_ids, list(self.target_classes.keys()))
        filtered_boxes = boxes[mask]
        filtered_conf = confidences[mask]
        filtered_class_ids = class_ids[mask]
        
        # Scale back to original image size
        x_factor = img_w / 640.0
        y_factor = img_h / 640.0
        
        nms_boxes = []
        nms_boxes_offset = []
        
        for i, row in enumerate(filtered_boxes):
            cx, cy, w, h = row
            left = int((cx - w / 2) * x_factor)
            top = int((cy - h / 2) * y_factor)
            width = int(w * x_factor)
            height = int(h * y_factor)
            
            nms_boxes.append([left, top, width, height])
            offset = int(filtered_class_ids[i] * 4096)
            nms_boxes_offset.append([left + offset, top + offset, width, height])
        
        # Apply NMS
        indices = cv2.dnn.NMSBoxes(nms_boxes_offset, filtered_conf.tolist(), 0.35, 0.45)
        
        detected_items = []
        max_area = img_w * img_h * 0.25
        
        if len(indices) > 0:
            for i in indices.flatten():
                box = nms_boxes[i]
                conf = filtered_conf[i]
                cls_id = filtered_class_ids[i]
                x, y, w, h = box[0], box[1], box[2], box[3]
                
                category = self.target_classes.get(cls_id, "unknown")
                if category not in self.valid_classes:
                    continue

                # Check for false positives
                overlap_flag = False
                box_area = w * h
                
                # Filter out boxes that are too large
                if box_area > max_area:
                    overlap_flag = True
                
                # Check intersection with face boxes
                for f_box in face_boxes:
                    face_area = f_box[2] * f_box[3]
                    intersect_area = self._calculate_intersection_area((x, y, w, h), f_box)
                    
                    if box_area > 0 and (intersect_area / box_area) > 0.3:
                        overlap_flag = True
                        break
                    if face_area > 0 and (intersect_area / face_area) > 0.4:
                        overlap_flag = True
                        break
                
                if not overlap_flag:
                    detected_items.append({"box": (x, y, w, h), "category": category})
                
        return detected_items

    def close(self):
        pass

print("FoodDetector class loaded successfully!")
print("\nDetectable Classes: Bottle, Lunch Box, Cup")
print("Model Format: YOLOv8 ONNX")
print("Inference Backend: ONNX Runtime (CPU)")

---

## Module 4: Performance Metrics & Logging

### Purpose:
Monitors and logs system performance metrics including:
- FPS tracking
- RAM/GPU usage
- Tampering events with timestamps
- Hardware specifications

In [ ]:
import csv
import os
import platform
import subprocess
import time
from datetime import datetime
import cv2
import psutil

try:
    import GPUtil
    GPU_AVAILABLE = True
except ImportError:
    GPU_AVAILABLE = False

class PerformanceEvaluator:
    """Tracks performance metrics and logs events."""
    
    def __init__(self, log_dir="data-logs", spec_file="hardware_specs.txt", 
                 log_file="hardware_logs.csv", tamper_file="tamper_events.csv"):
        self.log_dir = log_dir
        self.spec_path = os.path.join(log_dir, spec_file)
        self.log_path = os.path.join(log_dir, log_file)
        self.tamper_path = os.path.join(log_dir, tamper_file)

        os.makedirs(self.log_dir, exist_ok=True)

        self.fps_list = []
        self.fps = 0.0
        self.max_ram_used_gb = 0.0
        self.gpu_name = "N/A"
        self.max_gpu_load = 0.0
        self.max_gpu_mem = 0.0

        self._setup_tamper_csv()
        self._generate_specs_file()

    def _get_processor_name(self):
        try:
            if platform.system() == "Windows":
                return platform.processor()
            elif platform.system() == "Darwin":
                command = "sysctl -n machdep.cpu.brand_string"
                return subprocess.check_output(command, shell=True).decode().strip()
            elif platform.system() == "Linux":
                command = "cat /proc/cpuinfo | grep 'model name' | uniq"
                return subprocess.check_output(command, shell=True).decode().split(':')[1].strip()
        except Exception:
            return platform.machine()
        return "Unknown Processor"

    def _generate_specs_file(self):
        device_name = platform.node()
        os_platform = f"{platform.system()} {platform.release()}"
        processor = self._get_processor_name()
        cpu_cores = psutil.cpu_count(logical=True)
        total_memory_gb = round(psutil.virtual_memory().total / (1024 ** 3), 2)

        gpu_format = "N/A"
        if GPU_AVAILABLE:
            try:
                gpus = GPUtil.getGPUs()
                if gpus:
                    gpu_format = gpus[0].name
                    self.gpu_name = gpu_format
            except Exception:
                pass

        with open(self.spec_path, "a", encoding="utf-8") as f:
            f.write(f"\n--- New Session: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} ---\n")
            f.write(f"Device Name: {device_name}\n")
            f.write(f"OS Platform: {os_platform}\n")
            f.write(f"Processor: {processor}\n")
            f.write(f"CPU Cores: {cpu_cores}\n")
            f.write(f"Total Memory: {total_memory_gb} GB\n")
            f.write(f"GPU: {gpu_format}\n")

    def _setup_tamper_csv(self):
        if not os.path.isfile(self.tamper_path):
            with open(self.tamper_path, mode="w", newline="", encoding="utf-8") as file:
                writer = csv.writer(file)
                writer.writerow(["Tamper Timestamp", "Target Object Category", "Tampered or not"])

    def update_and_draw(self, img, loop_start_time):
        """Update metrics and draw FPS on image."""
        current_time = time.time()
        time_diff = current_time - loop_start_time

        if time_diff > 0:
            current_fps = 1.0 / time_diff
            self.fps = (self.fps * 0.9) + (current_fps * 0.1)  # Exponential moving average

        self.fps_list.append(self.fps)

        ram_used_gb = round(psutil.virtual_memory().used / (1024 ** 3), 2)
        if ram_used_gb > self.max_ram_used_gb:
            self.max_ram_used_gb = ram_used_gb

        if GPU_AVAILABLE:
            try:
                gpus = GPUtil.getGPUs()
                if gpus:
                    gpu = gpus[0]
                    self.gpu_name = gpu.name
                    if gpu.load * 100 > self.max_gpu_load:
                        self.max_gpu_load = round(gpu.load * 100, 1)
                    if gpu.memoryUsed > self.max_gpu_mem:
                        self.max_gpu_mem = gpu.memoryUsed
            except Exception:
                pass

        cv2.putText(
            img,
            f"FPS: {int(self.fps)}",
            (img.shape[1] - 150, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2,
            cv2.LINE_AA,
        )
        return img

    def log_tamper_event(self, category, tampered_status="YES"):
        """Log a tampering event."""
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        with open(self.tamper_path, mode="a", newline="", encoding="utf-8") as file:
            writer = csv.writer(file)
            writer.writerow([timestamp, category, tampered_status])

    def finalize_test_log(self, tampered_status):
        """Generate final performance report."""
        if self.fps_list:
            avg_fps = round(sum(self.fps_list) / len(self.fps_list), 1)
            best_fps = round(max(self.fps_list), 1)
            lowest_fps = round(min(self.fps_list), 1)
        else:
            avg_fps, best_fps, lowest_fps = 0.0, 0.0, 0.0

        tampered_str = "YES" if tampered_status else "NO"

        file_exists = os.path.isfile(self.log_path)
        with open(self.log_path, mode="a", newline="", encoding="utf-8") as file:
            writer = csv.writer(file)
            if not file_exists:
                writer.writerow([
                    "Average FPS", "Best FPS", "Lowest FPS",
                    "Tampered or not", "RAM used (GB)", "GPU_Name",
                    "GPU Load (%)", "GPU memory used (MB)",
                ])

            writer.writerow([
                avg_fps, best_fps, lowest_fps, tampered_str,
                f"{self.max_ram_used_gb} GB", self.gpu_name,
                f"{self.max_gpu_load}%", f"{self.max_gpu_mem} MB",
            ])

    def close(self):
        pass

print("PerformanceEvaluator class loaded successfully!")
print("\nTracked Metrics:")
print("  - FPS (frames per second)")
print("  - RAM usage")
print("  - GPU usage (if available)")
print("  - Tampering events with timestamps")
print("  - Hardware specifications")

---

## Main Pipeline: Complete System Integration

### System Architecture:

```
Video Input (Webcam)
    ↓
┌─────────────────────────────────────────────────┐
│  Hand Detection (MediaPipe)                     │
│  ↓                                              │
│  Face Authentication (InsightFace + MediaPipe)  │
│  ↓                                              │
│  Food Detection (YOLOv8 ONNX)                   │
│  ↓                                              │
│  Tampering Logic:                               │
│    IF hand touches food AND user NOT owner      │
│    THEN trigger alert                           │
└─────────────────────────────────────────────────┘
    ↓
┌─────────────────────────────────────────────────┐
│  Alert Actions:                                 │
│  - Save snapshot                                │
│  - Save 15s video buffer                        │
│  - Send Telegram notification                   │
│  - Log event to CSV                             │
└─────────────────────────────────────────────────┘
```

In [ ]:
import os

TELEGRAM_TOKEN = os.environ.get("TELEGRAM_TOKEN")
TELEGRAM_CHAT_ID = os.environ.get("TELEGRAM_CHAT_ID")

print("Telegram alerts use local environment variables when configured.")

---

## Dataset Testing Pipeline

### Purpose:
Test the food detection model on a batch of images without real-time camera input. Useful for benchmarking and evaluating model performance on static datasets.

In [ ]:
import os
import cv2
import time

def run_dataset_test():
    """
    Test detection model on static dataset images.
    
    Directory Structure Expected:
    test_dataset/
        images/
            image1.jpg
            image2.jpg
            ...
    """
    
    # Initialize components
    food_detector = FoodDetector()
    metrics_tracker = PerformanceEvaluator(
        log_dir="dataset-test-logs",
        log_file="dataset_hardware_metrics.csv",
        tamper_file="dataset_events.csv"
    )
    
    input_folder = "test_dataset/images"
    output_folder = "test_dataset/results"
    os.makedirs(output_folder, exist_ok=True)
    
    if not os.path.exists(input_folder):
        print(f"[-] Error: '{input_folder}' directory not found.")
        print(f"[*] Please create it and add test images.")
        return
        
    # Gather all images
    image_extensions = ('.jpg', '.jpeg', '.png', '.bmp')
    images = [f for f in os.listdir(input_folder) if f.lower().endswith(image_extensions)]
    
    print(f"[+] Found {len(images)} images. Starting benchmark test...")
    
    for idx, img_name in enumerate(images):
        img_path = os.path.join(input_folder, img_name)
        img = cv2.imread(img_path)
        if img is None:
            continue
            
        clean_img = img.copy()
        start_time = time.time()
        
        # Run detection
        detected_items = food_detector.process_and_draw(img, clean_img)
        
        # Draw results
        for item in detected_items:
            x, y, w, h = item["box"]
            category = item["category"]
            
            cv2.rectangle(img, (x, y), (x + w, y + h), (255, 100, 0), 2)
            cv2.putText(img, f"{category.upper()}", (x, y - 10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 100, 0), 2)
            
            metrics_tracker.log_tamper_event(category, tampered_status="DETECTED")
            
        # Update metrics
        metrics_tracker.update_and_draw(img, start_time)
        
        # Save output
        output_path = os.path.join(output_folder, f"evaluated_{img_name}")
        cv2.imwrite(output_path, img)
        print(f"[{idx + 1}/{len(images)}] Evaluated: {img_name}")
        
    # Generate report
    metrics_tracker.finalize_test_log(tampered_status=True)
    print("\n[SUCCESS] Dataset testing complete!")
    print(f"[->] Results saved to: {output_folder}")
    print(f"[->] Metrics saved to: dataset-test-logs/")

# Uncomment to run
# run_dataset_test()

print("Dataset testing function defined.")
print("\nTo run the dataset test, uncomment and execute: run_dataset_test()")

---

## Model Training Pipeline

### Purpose:
Train a custom YOLOv8 model for detecting food containers. This script:
1. Downloads the dataset from Roboflow (Cup, Bottle, Lunch Box)
2. Trains YOLOv8 Nano model
3. Exports to ONNX format for inference

In [ ]:
training_code = """
import os
from roboflow import Roboflow
from ultralytics import YOLO

roboflow_api_key = os.environ.get("ROBOFLOW_API_KEY")
if not roboflow_api_key:
    raise RuntimeError("Set ROBOFLOW_API_KEY before downloading the training dataset.")

rf = Roboflow(api_key=roboflow_api_key)
# Continue with the project/version download and YOLO training configuration.
"""
print("Model training code requires ROBOFLOW_API_KEY from the environment.")

---

## System Architecture Overview

### Technology Stack

| Component | Technology | Purpose |
|-----------|-----------|----------|
| **Face Authentication** | InsightFace + MediaPipe | Owner verification |
| **Hand Detection** | MediaPipe HandLandmarker | Gesture tracking |
| **Food Detection** | YOLOv8 ONNX | Container identification |
| **Inference** | ONNX Runtime | Cross-platform model serving |
| **Video Processing** | OpenCV (cv2) | Frame capture and rendering |
| **Alerts** | Telegram Bot API | Real-time notifications |
| **Metrics** | psutil, GPUtil | Hardware monitoring |

### Security Features

1. **Owner Authentication**: Only owner can touch items without triggering alert
2. **Hand Pose Tracking**: Detects hand proximity to food containers
3. **Event Logging**: CSV records of all tampering attempts
4. **Evidence Capture**: Snapshots and 15-second video buffers
5. **Real-time Alerts**: Telegram notifications with photo/video evidence
6. **Performance Monitoring**: Hardware metrics for system auditing

### Processing Pipeline Flow

```
Frame Input
    ↓
MediaPipe Preprocessing
    ↓
┌─────────────────────────────────────┐
│ Parallel Processing:                │
├─────────────────────────────────────┤
│ 1. Hand Detection (21 landmarks)    │
│ 2. Face Detection (mediapipe)       │
│    └→ Face Embedding (insightface)  │
│ 3. Food Detection (YOLOv8 ONNX)     │
└─────────────────────────────────────┘
    ↓
Tampering Logic Check
    ↓
Render Annotations
    ↓
Display & Log Metrics
```

---

## Usage Guide

### Quick Start

1. **Setup**: Install requirements and download models
2. **Prepare**: Create `user.jpg` (reference photo of owner's face)
3. **Run**: Execute `python main.py` to start the real-time pipeline
4. **Monitor**: Watch for red boxes indicating tampering attempts
5. **Review**: Check `evidence/` folder for snapshots and videos
6. **Analyze**: Review `data-logs/` CSV files for event history

### Running Different Modes

**Mode 1: Real-time Monitoring**
```bash
python main.py
```

**Mode 2: Dataset Benchmark**
```bash
python test_pipeline.py
```

**Mode 3: Model Training** (requires GPU)
```bash
python data_build.py
```

---

## Performance Metrics

### Expected Performance (CPU-based)

- **FPS**: 10-20 FPS on modern CPU
- **Latency**: 50-100ms per frame
- **RAM Usage**: 2-4 GB
- **GPU Memory** (if available): 100-500 MB

### Logged Metrics

```csv
Average FPS, Best FPS, Lowest FPS, Tampered or not, RAM used (GB), GPU_Name, GPU Load (%), GPU memory used (MB)
15.2, 18.5, 12.1, YES, 3.8 GB, N/A, N/A, N/A
```

### Event Log Example

```csv
Tamper Timestamp, Target Object Category, Tampered or not
2026-08-30 14:32:15, Bottle, YES
2026-08-30 14:35:42, Cup, YES
2026-08-30 14:38:09, Lunch Box, YES
```

---

## Troubleshooting

### Common Issues

| Issue | Solution |
|-------|----------|
| Webcam not opening | Check USB connection, try `cv2.CAP_DSHOW` or `cv2.CAP_V4L2` |
| Model file not found | Run `download_models()` to fetch from MediaPipe |
| Face not detected | Ensure good lighting, face visible in frame |
| Low FPS | Reduce input resolution or use GPU |
| Telegram alerts not working | Check token, chat ID, and internet connection |
| Memory leak | Ensure `close()` methods are called for all trackers |

---

## Project Structure

```
ezyZip/
├── main.py                    # Main real-time pipeline
├── test_pipeline.py           # Dataset testing mode
├── data_build.py              # Model training (Roboflow → YOLOv8)
├── utils_download.py          # Download pre-trained models
├── module_hands.py            # HandTracker class
├── module_faces.py            # FaceTracker class
├── module_objects.py          # FoodDetector class
├── module_metrics.py          # PerformanceEvaluator class
├── evaluate_metrics.py        # Metrics evaluation utilities
├── requirements.txt           # Python dependencies
├── Complete_Project.ipynb     # This notebook
├── hand_landmarker.task       # MediaPipe hand model
├── face_landmarker.task       # MediaPipe face model
├── best.onnx                  # YOLOv8 custom model
├── yolov8n.pt                 # YOLOv8 Nano pretrained
├── evidence/                  # Snapshots and videos
├── data-logs/                 # Hardware and event logs
└── Combined_Cup_Bottle_lunchBox-2/  # Training dataset
```

---

## Conclusion

A sophisticated AI-powered security system that combines multiple state-of-the-art deep learning models to protect food items from unauthorized tampering. The system demonstrates:

✅ Real-time multi-modal AI processing
✅ Owner-aware authentication
✅ Accurate object detection and hand pose tracking
✅ Event logging and evidence capture
✅ Scalable architecture with modular components
✅ Cross-platform compatibility (CPU-based inference)

The project is suitable for deployment in kitchens, offices, and other shared spaces where food security is critical.

---

**Project Author**: Team Jonaki
**Last Updated**: 2026-08-30
**License**: Proprietary